In [28]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset, Dataset
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, classification_report

In [5]:
df = pd.DataFrame(pd.read_csv("/content/draft_2.csv"))

In [7]:
df.columns

Index(['text', 'quality', 'avg_rating', 'kind', 'gl_label', 'inc_label',
       'toxicity', 'threat', 'profanity', 'toxic_label', 'threat_label',
       'profanity_label'],
      dtype='object')

In [20]:
df['is_incomplete'] = (df['inc_label'] == 'reproduce').astype(float)
df['is_bad_quality'] = (df['gl_label'] == 'not good').astype(float)

target_cols = [
    'toxic_label', 'threat_label', 'profanity_label',
    'is_incomplete', 'is_bad_quality'
]
df['labels'] = df[target_cols].values.tolist()
print(df[['text', 'labels']].head())

                                                text  \
0  What can you tell me about Aharon Barak's Cons...   
1  Compose an analysis of the cultural significan...   
2                                  what are nanobots   
3  what is 198555 x 725894?\nOnly mathematical sy...   
4  You are an expert programmer. Optimize this co...   

                      labels  
0  [0.0, 0.0, 0.0, 0.0, 0.0]  
1  [0.0, 0.0, 0.0, 0.0, 0.0]  
2  [0.0, 0.0, 0.0, 1.0, 1.0]  
3  [0.0, 0.0, 0.0, 0.0, 0.0]  
4  [0.0, 0.0, 0.0, 0.0, 0.0]  


In [23]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def preprocess_function(examples):
    # Tokenize the text
    encoding = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    # Ensure labels are in the correct float format for Multi-Label loss
    encoding["labels"] = [list(map(float, l)) for l in examples["labels"]]
    return encoding
# Convert to HF Dataset
dataset = Dataset.from_pandas(df[['text', 'labels']])
# Split into train and test (80/20)
dataset = dataset.train_test_split(test_size=0.2)

tokenized_ds = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/11167 [00:00<?, ? examples/s]

Map:   0%|          | 0/2792 [00:00<?, ? examples/s]

In [24]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5,
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
training_args = TrainingArguments(
    output_dir="./toxicity_quality_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.258197,0.185120
2,0.168678,0.173585
3,0.113256,0.173920


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2094, training_loss=0.16885072750029614, metrics={'train_runtime': 914.9174, 'train_samples_per_second': 36.616, 'train_steps_per_second': 2.289, 'total_flos': 2203680222671616.0, 'train_loss': 0.16885072750029614, 'epoch': 3.0})

In [29]:
from sklearn.metrics import classification_report
import numpy as np

# 1. Get predictions on your test set
predictions = trainer.predict(tokenized_ds["test"])
preds = predictions.predictions
labels = predictions.label_ids

# 2. Apply Sigmoid to turn raw scores into 0 or 1
# (Since it's multi-label, we check if each score is > 0)
sig_preds = 1 / (1 + np.exp(-preds)) # Manual sigmoid
binary_preds = (sig_preds > 0.5).astype(int)

# 3. See how it performed on each of the 5 categories
target_names = ['Toxic', 'Threat', 'Profanity', 'Incomplete', 'Bad Quality']
print(classification_report(labels, binary_preds, target_names=target_names))

              precision    recall  f1-score   support

       Toxic       0.89      0.91      0.90       812
      Threat       0.88      0.90      0.89       617
   Profanity       0.82      0.84      0.83       473
  Incomplete       0.98      0.84      0.90      1240
 Bad Quality       0.99      0.86      0.92      1207

   micro avg       0.93      0.87      0.90      4349
   macro avg       0.91      0.87      0.89      4349
weighted avg       0.93      0.87      0.90      4349
 samples avg       0.35      0.35      0.35      4349



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [30]:
# Define your path in Drive
# You can change 'my_bert_model' to any folder name you like
drive_path = "/content/drive/MyDrive/Audios/toxicity_quality_bert_model"

# Save using the trainer (this saves the model weights and config)
trainer.save_model(drive_path)

# Save the tokenizer explicitly (best practice)
tokenizer.save_pretrained(drive_path)

print(f"Model successfully saved to: {drive_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to: /content/drive/MyDrive/Audios/toxicity_quality_bert_model
